In [ ]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

PROCESSING=False 
PROCESSING=True #set to True if using Turbulence-Removed Binary Arrays
Processed_string = GetProcessedString(PROCESSING=PROCESSING)

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=2)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation_DivideMassFlux",
                                dtype='int32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#CALCULATING ENTRAINMENT
def GetVariable_T(t, varName, cache=None): #*MassFlux
    """Load variable for a given timestep, using cached open HDF5 files if available."""
    if cache is None:
        cache = {}

    timeString = ModelData.timeStrings[t]

    # If file for this timestep already in cache, reuse it
    if timeString in cache:
        f = cache[timeString]
    else:
        # Otherwise, open new file and cache it
        f = DataManager.GetTimestepData_V2(DataManager.inputDataDirectory, timeString)
        cache[timeString] = f

    # Load the desired variable from the open file (lazy read)
    if varName in f.keys():
        output = f[varName][:]  # read actual data
    else:
        # fallback for derived variables
        output = CallVariable(ModelData, DataManager, timeString, variableName=varName)
    return output

def SafeDivide(numerator, denominator):
    # 1. Create an array of NaNs with the same shape as the numerator
    result = np.full(numerator.shape, np.nan, dtype=np.float32)
    
    # 2. Find where we can safely divide
    nonzero = (denominator != 0)
    
    # 3. Only fill the "nonzero" indices with the division result
    result[nonzero] = numerator[nonzero] / denominator[nonzero]
    
    return result
    
# def LoadMassFlux(t): #*MassFlux
#     """
#     lagrangian version
#     """
    
#     updraftType = "g" 
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux_g = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     updraftType = "c" 
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux_c = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     return MassFlux_g, MassFlux_c

def LoadMassFlux(t):
    """
    eulerian version
    """
    
    timeString=ModelData.timeStrings[t]

    varName='rho'
    rho = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='winterp'
    w = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='A_g'
    A_g = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    varName='A_c'
    A_c = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    
    MassFlux_g = rho*(A_g*1)*w
    MassFlux_c = rho*(A_c*1)*w
    return MassFlux_g,MassFlux_c

def DivideByMassFlux(t, varName,
                     MassFlux_g, MassFlux_c):
    variable = GetVariable_T(t, varName)
    MassFlux = MassFlux_g if "g" in varName else MassFlux_c
    variable_DivideMassFlux = SafeDivide(variable, MassFlux)
    return variable_DivideMassFlux

In [ ]:
def GetVarNames_Entrainment(Processed_string):
    varNames = [f"{Processed_string}Entrainment_g",
                f"{Processed_string}Entrainment_c"]
    
    varNames += [f"{Processed_string}TransferEntrainment_g",
                f"{Processed_string}TransferEntrainment_c"]
    return varNames

def GetVarNames_Detrainment(Processed_string):
    varNames = [f"{Processed_string}Detrainment_g",
                f"{Processed_string}Detrainment_c"]
    
    varNames += [f"{Processed_string}TransferDetrainment_g",
                f"{Processed_string}TransferDetrainment_c"]
    return varNames

def AddDivideMassFluxString(varName):
    parts = varName.rsplit('_', 1)
    varName_DivideMassFlux = f"{parts[0]}_DivideMassFlux_{parts[1]}"
    return varName_DivideMassFlux

In [ ]:
def RunCalculation_Entrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Entrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

def RunCalculation_Detrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Detrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

In [ ]:
##############################################
#RUNNING

In [ ]:
#running calculation
for t in tqdm(loop_elements, total=len(loop_elements)):
    # if np.mod(t,1)==0: print(f'Current time {t}')
    if t == ModelData.Ntime-1:
        continue

    #loading MassFlux
    [MassFlux_g, MassFlux_c] =  LoadMassFlux(t)

    #calculating variables
    outputDictionary_Entrainment_DivideMassFlux = RunCalculation_Entrainment(t, Processed_string,
                                                                         MassFlux_g, MassFlux_c)
    outputDictionary_Detrainment_DivideMassFlux = RunCalculation_Detrainment(t, Processed_string,
                                                                             MassFlux_g, MassFlux_c)
    
    #outputting
    timeString = ModelData.timeStrings[t]
    
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Entrainment_DivideMassFlux, dataName=f"{Processed_string}Entrainment_DivideMassFlux",dtype='float32')

    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Detrainment_DivideMassFlux, dataName=f"{Processed_string}Detrainment_DivideMassFlux",dtype='float32')

In [ ]:
########################
#TESTING

In [ ]:
# #FUNCTIONS

# import sys
# path=os.path.join(mainCodeDirectory,'Functions/')
# sys.path.append(path)

# import NumericalFunctions
# from NumericalFunctions import * # import NumericalFunctions 
# import PlottingFunctions
# from PlottingFunctions import * # import PlottingFunctions

# # # Get all functions in NumericalFunctions
# # import inspect
# # functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# # functions

# #FUNCTIONS

# #APPLYING ENTRAINMENT CONSTANT
# entrainmentConstant = DataManager.LoadCalculations(
#     DataManager.outputDataDirectory.replace('EntrainmentCalculation_DivideMassFlux','EntrainmentCalculation'),
#     dataName="EntrainmentConstant",
#     verbose=False,
# )["entrainmentConstant"]
# def ApplyEntrainmentConstant(variable):
#     """
#     Multiply each array in the input dictionary by the 1D entrainment constant profile.
#     Returns the processed arrays in the same order as the input dictionary.
#     """
    
#     # Return arrays in the same order as input
#     return variable*entrainmentConstant[:, np.newaxis, np.newaxis]

# #APPLYING MASS FLUX CONSTANT
# massFluxConstant = DataManager.LoadCalculations(
#     os.path.join(os.path.dirname(DataManager.outputDataDirectory), "MassFluxCalculation"),
#     dataName="MassFluxConstant",
#     verbose=False,
# )["massFluxConstant"]
# def ApplyMassFluxConstant(variable): #*MassFlux
#     """
#     Multiply each array in the input dictionary by the 1D mass flux constant profile.
#     Returns the processed arrays in the same order as the input dictionary.
#     """

#     # Return arrays in the same order as input
#     return variable * massFluxConstant[:, np.newaxis, np.newaxis]

# def LoadMassFlux_lagrangian(t,updraftType="c"): #*MassFlux
#     """
#     lagrangian version
#     """
    
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     return MassFlux

# def CalculateMassFlux_eulerian(t,updraftType="c"):
    
#     timeString=ModelData.timeStrings[t]

#     varName='rho'
#     rho = CallVariable(ModelData, DataManager, timeString, 
#                                                   variableName=varName)
    
#     varName='winterp'
#     w = CallVariable(ModelData, DataManager, timeString, 
#                                                   variableName=varName)
#     varName=f'A_{updraftType}'
#     A = CallVariable(ModelData, DataManager, timeString, 
#                                                   variableName=varName)

#     MassFlux = rho*(A*1)*w
#     return MassFlux

# def GetData_multipletimes(ts, updraftType="c"):

#     varName = f"{Processed_string}Entrainment_{updraftType}"
    
#     # Get shape from a sample
#     sample = GetVariable_T(ts[0], varName)
#     Nz = sample.shape[0]

#     # Pre-allocate (time, z)
#     variable_array = np.full((len(ts), Nz), np.nan)
#     MassFlux_lagrangian_array = np.full((len(ts), Nz), np.nan)
#     MassFlux_eulerian_array = np.full((len(ts), Nz), np.nan)

#     for i, t in tqdm(enumerate(ts), total=len(ts)):
#         v = GetVariable_T(t, varName)
#         v = ApplyEntrainmentConstant(v)
        
#         mfl = LoadMassFlux_lagrangian(t,updraftType)
#         mfl = ApplyMassFluxConstant(mfl)
#         mfe = CalculateMassFlux_eulerian(t,updraftType)

#         # --- Average over x,y (axis 1,2) ---
#         variable_array[i] = np.nanmean(v, axis=(1, 2))
#         MassFlux_lagrangian_array[i] = np.nanmean(mfl, axis=(1, 2))
#         MassFlux_eulerian_array[i] = np.nanmean(mfe, axis=(1, 2))

#     return variable_array, MassFlux_lagrangian_array, MassFlux_eulerian_array

In [ ]:
# def MakePlot_line(variable, MassFlux, MassFlux_eulerian, 
#                       type="singletime"):

#     # Create 2x3 layout
#     fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
#     maxZ = 14

#     # Axis selection
#     if type == "singletime":
#         axis = (1, 2)
#     elif type == "multipletimes":
#         axis = (0)

#     # Core shared variables
#     a = np.nanmean(variable, axis=axis)
#     b = np.nanmean(MassFlux, axis=axis)

#     mask = ModelData.zh > maxZ
#     a[mask], b[mask] = np.nan, np.nan

#     # -------------------------
#     # TOP ROW: LAGRANGIAN
#     # -------------------------
#     c = np.nanmean(SafeDivide(variable, MassFlux), axis=axis)
#     c[mask] = np.nan

#     ax1, ax2, ax3 = axes[0]

#     ax1.plot(a, ModelData.zh, color='blue')
#     ax1.set_title('Lagrangian Entrainment')
#     ax1.set_ylabel('z (km)')

#     ax2.plot(b, ModelData.zh, color='orange')
#     ax2.set_title('Lagrangian MassFlux')

#     ax3.plot(c, ModelData.zh, color='blue')
#     ax3.set_title('Mean of Ratio')

#     # -------------------------
#     # BOTTOM ROW: EULERIAN
#     # -------------------------
#     d = np.nanmean(MassFlux_eulerian, axis=axis)
#     e = np.nanmean(SafeDivide(variable, MassFlux_eulerian), axis=axis)

#     d[mask], e[mask] = np.nan, np.nan

#     ax4, ax5, ax6 = axes[1]

#     ax4.plot(a, ModelData.zh, color='blue')
#     ax4.set_title('Lagrangian Entrainment')
#     ax4.set_ylabel('z (km)')

#     ax5.plot(d, ModelData.zh, color='orange')
#     ax5.set_title('Eulerian MassFlux')

#     ax6.plot(e, ModelData.zh, color='blue')
#     ax6.set_title('Mean of Ratio')

#     # Common formatting
#     for ax in axes.flatten():
#         ax.set_ylim(0, maxZ)

#     for ax in [ax3, ax6]:
#         ax.set_xlabel(r"$(s^{-1})$")

#     apply_scientific_notation(axes.flatten())

#     plt.tight_layout()

#     # --- Match x-limits within each column ---
#     for col in range(3):
#         top_ax = axes[0, col]
#         bottom_ax = axes[1, col]
    
#         # Get current limits
#         xmin_top, xmax_top = top_ax.get_xlim()
#         xmin_bot, xmax_bot = bottom_ax.get_xlim()
    
#         # Combine
#         xmin = np.nanmin([xmin_top, xmin_bot])
#         xmax = np.nanmax([xmax_top, xmax_bot])
    
#         # Apply to both
#         top_ax.set_xlim(xmin, xmax)
#         bottom_ax.set_xlim(xmin, xmax)

#     fig.suptitle(
#         f'Entrainment / MassFlux Comparison (Lagrangian vs Eulerian)\n'
#         f'Averaged over {len(ts)*ModelData.dt/60:.0f} minutes',
#         fontsize=16,
#         fontweight='bold',
#         y=1.04
#     )
#     return fig

# def MakePlot_contour(variable, MassFlux, MassFlux_eulerian,
#                      cmap='turbo'):

#     fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
#     maxZ = 14

#     t_axis = np.arange(variable.shape[0])
#     z_axis = ModelData.zh

#     # Mask
#     mask = z_axis > maxZ

#     # -------------------------
#     # TOP ROW: LAGRANGIAN
#     # -------------------------
#     a = variable.copy()
#     b = MassFlux.copy()
#     c = SafeDivide(variable, MassFlux)

#     a[:, mask] = np.nan
#     b[:, mask] = np.nan
#     c[:, mask] = np.nan

#     # -------------------------
#     # BOTTOM ROW: EULERIAN
#     # -------------------------
#     d = MassFlux_eulerian.copy()
#     e = SafeDivide(variable, MassFlux_eulerian)

#     d[:, mask] = np.nan
#     e[:, mask] = np.nan

#     # -------------------------
#     # Shared contour levels per column
#     # -------------------------
#     # levels1 = np.linspace(np.nanmin(a), np.nanmax(a), 20)
#     # levels2 = np.linspace(np.nanmin([b, d]), np.nanmax([b, d]), 20)
#     # levels3 = np.linspace(np.nanmin([c, e]), np.nanmax([c, e]), 20)

#     percentile_min = 1;percentile_max = 99
#     levels1 = np.linspace(
#         np.nanpercentile(a, percentile_min),
#         np.nanpercentile(a, percentile_max),
#         20
#     )
#     levels2 = np.linspace(
#         np.nanpercentile(np.concatenate([b.ravel(), d.ravel()]), percentile_min),
#         np.nanpercentile(np.concatenate([b.ravel(), d.ravel()]), percentile_max),
#         20
#     )
#     levels3 = np.linspace(
#         np.nanpercentile(np.concatenate([c.ravel(), e.ravel()]), percentile_min),
#         np.nanpercentile(np.concatenate([c.ravel(), e.ravel()]), percentile_max),
#         20
#     )

#     ax1, ax2, ax3 = axes[0]
#     ax4, ax5, ax6 = axes[1]

#     # Column 1
#     im1 = ax1.contourf(t_axis, z_axis, a.T, cmap=cmap, levels=levels1, extend="both")
#     ax1.set_title('Lagrangian Entrainment')
#     ax1.set_ylabel('z (km)')
#     cbar1 = fig.colorbar(im1, ax=ax1)
    
#     im4 = ax4.contourf(t_axis, z_axis, a.T, cmap=cmap, levels=levels1, extend="both")
#     ax4.set_title('Lagrangian Entrainment')
#     ax4.set_ylabel('z (km)')
#     cbar4 = fig.colorbar(im4, ax=ax4)
    
#     # Column 2
#     im2 = ax2.contourf(t_axis, z_axis, b.T, cmap=cmap, levels=levels2, extend="both")
#     ax2.set_title('Lagrangian MassFlux')
#     cbar2 = fig.colorbar(im2, ax=ax2)
    
#     im5 = ax5.contourf(t_axis, z_axis, d.T, cmap=cmap, levels=levels2, extend="both")
#     ax5.set_title('Eulerian MassFlux')
#     cbar5 = fig.colorbar(im5, ax=ax5)
    
#     # Column 3
#     im3 = ax3.contourf(t_axis, z_axis, c.T, cmap=cmap, levels=levels3, extend="both")
#     ax3.set_title('Mean of Ratio')
#     cbar3 = fig.colorbar(im3, ax=ax3)
    
#     im6 = ax6.contourf(t_axis, z_axis, e.T, cmap=cmap, levels=levels3, extend="both")
#     ax6.set_title('Mean of Ratio')
#     cbar6 = fig.colorbar(im6, ax=ax6)

#     # Formatting
#     for ax in axes.flatten():
#         ax.set_ylim(0, maxZ)
#         ax.set_xlabel('Time index')

#     plt.tight_layout()

#     cbars = [cbar1, cbar2, cbar3, cbar4, cbar5, cbar6]

#     for cbar in cbars:
#         formatter = ticker.ScalarFormatter(useMathText=True)
#         formatter.set_scientific(True)
#         formatter.set_powerlimits((0, 0))
#         cbar.formatter = formatter
#         cbar.update_ticks()

#     return fig

In [ ]:
# #MULTIPLE TIMESTEPS

# # # Calculating
# step = 1 if ModelData.t_res == "5min" else 5
# # ts = range(120*step, 122*step)
# # ts = range(90*step, 130*step) 
# # ts = np.arange(ModelData.Ntime) 
# ts = np.arange(50*step,110*step) 

# variable,MassFlux_lagrangian,MassFlux_eulerian = GetData_multipletimes(ts,updraftType="c")

# fig = MakePlot_line(variable, MassFlux_lagrangian, MassFlux_eulerian,
#                         type="multipletimes")

# fig = MakePlot_contour(variable, MassFlux_lagrangian, MassFlux_eulerian)

In [ ]:
# #MULTIPLE TIMESTEPS

# # # Calculating
# step = 1 if ModelData.t_res == "5min" else 5
# # ts = range(120*step, 122*step)
# # ts = range(90*step, 130*step) 
# # ts = np.arange(ModelData.Ntime) 
# ts = np.arange(50*step,110*step) 

# variable,MassFlux_lagrangian,MassFlux_eulerian = GetData_multipletimes(ts,updraftType="g")

# fig = MakePlot_line(variable, MassFlux_lagrangian, MassFlux_eulerian,
#                         type="multipletimes")

# fig = MakePlot_contour(variable, MassFlux_lagrangian, MassFlux_eulerian)

In [ ]:
# #old
# oneTime=True
# if oneTime==True:
#     t=96
#     t*=5 if len(ModelData.zh) != 34 else 1
#     [MF_g_lagrangian,MF_c_lagrangian] = LoadMassFlux_lagrangian(t=t)
#     [A_g_lagrangian,A_c_lagrangian] = GetACount(t)
#     [MF_g_eulerian,MF_c_eulerian, A_g_eulerian,A_c_eulerian] = LoadMassFlux_eulerian(t=t)
# # else:
# #     tStart = 110 
# #     tEnd = 120  # inclusive
# #     tStart *= 5 if len(ModelData.zh) != 34 else 1
# #     tEnd *= 5 if len(ModelData.zh) != 34 else 1
    
# #     Nt = tEnd - tStart + 1
# #     Nz = ModelData.Nzh  
    
# #     MF_g_lagrangian = np.zeros((Nt, Nz))
# #     MF_c_lagrangian = np.zeros((Nt, Nz))
# #     MF_g_eulerian  = np.zeros((Nt, Nz))
# #     MF_c_eulerian  = np.zeros((Nt, Nz))
# #     A_g_lagrangian = np.zeros((Nt, Nz))
# #     A_c_lagrangian = np.zeros((Nt, Nz))
# #     A_g_eulerian  = np.zeros((Nt, Nz))
# #     A_c_eulerian  = np.zeros((Nt, Nz))
    
# #     for ti, t in enumerate(tqdm(range(tStart, tEnd + 1))):
# #         [mf_g_l, mf_c_l] = LoadMassFlux_lagrangian(t=t)
# #         [a_g_l, a_c_l]   = GetACount(t)
# #         [mf_g_e, mf_c_e, a_g_e, a_c_e] = LoadMassFlux_eulerian(t=t)
    
# #         MF_g_lagrangian[ti, :] = np.mean(mf_g_l, axis=(1,2))
# #         MF_c_lagrangian[ti, :] = np.mean(mf_c_l, axis=(1,2))
# #         MF_g_eulerian[ti, :]   = np.mean(mf_g_e, axis=(1,2))
# #         MF_c_eulerian[ti, :]   = np.mean(mf_c_e, axis=(1,2))
    
# #         A_g_lagrangian[ti, :] = np.sum(a_g_l, axis=(1,2))
# #         A_c_lagrangian[ti, :] = np.sum(a_c_l, axis=(1,2))
# #         A_g_eulerian[ti, :]   = np.sum(a_g_e, axis=(1,2))
# #         A_c_eulerian[ti, :]   = np.sum(a_c_e, axis=(1,2))

# fig, axes = plt.subplots(2, 2,figsize=(8,8))
# fig.subplots_adjust(hspace=0.3)

# meanAxis = (1,2) if oneTime is True else 0
# ax = axes[0, 0]
# a = np.mean(MF_g_lagrangian, axis=meanAxis)
# ax.plot(a, ModelData.zh, color='green', linestyle="--", label='lagrangian')
# b = np.mean(MF_g_eulerian, axis=meanAxis)
# ax.plot(b, ModelData.zh, color='green', linestyle="-", label='eulerian')
# ax.set_xlabel("VMF_g"+r" $(kg/m^2/s)$"); ax.set_ylabel("z (km)")
# ax.legend(fontsize=9)
# ax.set_title("mean of VMF at each level",fontsize=10)

# ax = axes[0, 1]
# a = np.mean(MF_c_lagrangian, axis=meanAxis)
# ax.plot(a, ModelData.zh, color='blue', linestyle="--", label='lagrangian')
# b = np.mean(MF_c_eulerian, axis=meanAxis)
# ax.plot(b, ModelData.zh, color='blue', linestyle="-", label='eulerian')
# ax.set_xlabel("VMF_c"+r" $(kg/m^2/s)$")
# ax.legend(fontsize=9)
# ax.set_title("mean of VMF at each level",fontsize=10)

# ax = axes[1, 0]
# a = np.sum(A_g_lagrangian, axis=meanAxis)
# ax.plot(a, ModelData.zh, color='green', linestyle="--", label='lagrangian')
# b = np.sum(A_g_eulerian, axis=meanAxis)
# ax.plot(b, ModelData.zh, color='green', linestyle="-", label='eulerian')
# ax.set_xlabel("A_g"); ax.set_ylabel("z (km)")
# ax.legend(fontsize=9)
# ax.set_title("sum of A_g at each level",fontsize=10)

# ax = axes[1, 1]
# a = np.sum(A_c_lagrangian, axis=meanAxis)
# ax.plot(a, ModelData.zh, color='blue', linestyle="--", label='lagrangian')
# b = np.sum(A_c_eulerian, axis=meanAxis)
# ax.plot(b, ModelData.zh, color='blue', linestyle="-", label='eulerian')
# ax.set_xlabel("A_c")
# ax.legend(fontsize=9)
# ax.set_title("sum of A_c at each level",fontsize=10)

# for ax in axes.ravel():
#     ax.set_ylim(0,20)